# MCP 工具版本：Schema 协商、适配与契约测试

**面试问题：工具 Schema 从 v1 升到 v2 时，怎样避免旧 Agent 静默调用错误？**

## 回答主线

1. MCP 工具描述是模型和服务之间的可执行合同，字段名、必填项、类型和枚举变化都可能破坏旧客户端。
2. 服务端应暴露稳定版本或能力协商，不能原地修改同名工具后假设所有 Agent 会自动适配。
3. 契约测试要覆盖至少一个合法样本和类型缺失、未知字段、枚举越界等反例。
4. 兼容适配器可以为旧请求显式补齐有业务依据的字段，并记录迁移来源。
5. 不确定字段不能静默猜默认值，尤其货币、租户和权限范围。
6. 发布流程应先跑客户端×服务端兼容矩阵，再灰度切换默认版本。

## 真实案例

发票工具 v1 接收 customer_id 与 amount，默认人民币；v2 将 currency 设为必填枚举并新增可选 memo。六个客户端请求覆盖旧合法调用、新合法调用、金额类型错误、缺少货币、未知字段和不支持货币。我们手写 JSON Schema 子集验证器和 v1→v2 适配器。数据是离线脱敏教学样本，只用于解释机制，不能宣称线上收益。

### 输入预览：两版 Schema 与六个请求

In [1]:
schemas = {  # 定义发票工具两个版本的精简 JSON Schema。
    "v1": {"required": ["customer_id", "amount"], "properties": {"customer_id": {"type": "string"}, "amount": {"type": "number"}}, "additionalProperties": False},  # v1 依赖服务端人民币默认值。
    "v2": {"required": ["customer_id", "amount", "currency"], "properties": {"customer_id": {"type": "string"}, "amount": {"type": "number"}, "currency": {"type": "string", "enum": ["CNY", "USD"]}, "memo": {"type": "string"}}, "additionalProperties": False},  # v2 明确货币并允许备注。
}  # 完成 Schema 注册表。
requests = [  # 构造六个客户端版本与负载组合。
    {"id": "C1", "client": "v1", "payload": {"customer_id": "U10", "amount": 299.0}},  # 合法旧客户端请求。
    {"id": "C2", "client": "v2", "payload": {"customer_id": "U11", "amount": 89.0, "currency": "CNY", "memo": "补开发票"}},  # 合法新客户端请求。
    {"id": "C3", "client": "v2", "payload": {"customer_id": "U12", "amount": "99", "currency": "CNY"}},  # 金额类型错误。
    {"id": "C4", "client": "v2", "payload": {"customer_id": "U13", "amount": 120.0}},  # v2 缺少货币。
    {"id": "C5", "client": "v2", "payload": {"customer_id": "U14", "amount": 55.0, "currency": "USD", "debug": True}},  # 含未知字段。
    {"id": "C6", "client": "v2", "payload": {"customer_id": "U15", "amount": 66.0, "currency": "EUR"}},  # 枚举越界。
]  # 完成契约样本。
print("请求  client  payload")  # 输出输入表头。
for request in requests:  # 逐请求展示版本和负载。
    print(f"{request['id']}   {request['client']:<6} {request['payload']}")  # 展示六类兼容情况。

请求  client  payload
C1   v1     {'customer_id': 'U10', 'amount': 299.0}
C2   v2     {'customer_id': 'U11', 'amount': 89.0, 'currency': 'CNY', 'memo': '补开发票'}
C3   v2     {'customer_id': 'U12', 'amount': '99', 'currency': 'CNY'}
C4   v2     {'customer_id': 'U13', 'amount': 120.0}
C5   v2     {'customer_id': 'U14', 'amount': 55.0, 'currency': 'USD', 'debug': True}
C6   v2     {'customer_id': 'U15', 'amount': 66.0, 'currency': 'EUR'}


## Baseline 基线：服务端直接替换成 v2

In [2]:
def naive_required_check(payload, schema):  # 只检查必填字段的朴素服务端。
    missing = [field for field in schema["required"] if field not in payload]  # 计算缺失字段。
    return len(missing) == 0, missing  # 返回是否通过和缺失列表。

baseline_rows = []  # 收集所有请求直接打到 v2 的结果。
for request in requests:  # 遍历旧新客户端请求。
    passed, missing = naive_required_check(request["payload"], schemas["v2"])  # 仅用新版必填字段校验。
    baseline_rows.append({"id": request["id"], "passed": passed, "missing": missing})  # 保存基线结论。
print("请求  直接v2通过  缺失字段")  # 输出破坏性升级结果表头。
for row in baseline_rows:  # 逐请求展示基线判断。
    print(f"{row['id']}    {str(row['passed']):<9} {row['missing']}")  # 展示 C1 旧请求被破坏而 C3/C5/C6 被误放行。

请求  直接v2通过  缺失字段
C1    False     ['currency']
C2    True      []
C3    True      []
C4    False     ['currency']
C5    True      []
C6    True      []


### 核心实现：JSON Schema 子集验证与显式版本适配

In [3]:
def validate(payload, schema):  # 手写教学版 Schema 验证器。
    errors = []  # 收集全部契约错误。
    for field in schema["required"]:  # 检查每个必填字段。
        if field not in payload:  # 当前负载缺少必填字段。
            errors.append(f"missing:{field}")  # 保存缺失错误。
    if not schema.get("additionalProperties", True):  # Schema 禁止未知字段时执行白名单检查。
        for field in payload:  # 遍历请求实际字段。
            if field not in schema["properties"]:  # 字段未出现在属性定义中。
                errors.append(f"unknown:{field}")  # 保存未知字段错误。
    python_types = {"string": str, "number": (int, float)}  # 建立 JSON 类型到 Python 类型的最小映射。
    for field, value in payload.items():  # 对存在且已知的字段检查类型和枚举。
        rule = schema["properties"].get(field)  # 读取字段规则。
        if rule is None:  # 未知字段已在上一步记录。
            continue  # 跳过无规则字段的后续检查。
        expected_type = python_types[rule["type"]]  # 读取期望 Python 类型。
        type_ok = isinstance(value, expected_type) and not (rule["type"] == "number" and isinstance(value, bool))  # 排除 bool 被当作整数的边界。
        if not type_ok:  # 实际类型不符合 Schema。
            errors.append(f"type:{field}")  # 保存类型错误。
        if "enum" in rule and value not in rule["enum"]:  # 检查枚举值是否受支持。
            errors.append(f"enum:{field}")  # 保存枚举错误。
    return len(errors) == 0, errors  # 返回完整验证结论。

def adapt_v1_to_v2(payload, tenant_currency):  # 把明确属于 v1 的请求迁移到 v2。
    adapted = payload.copy()  # 复制旧请求避免修改调用方对象。
    adapted["currency"] = tenant_currency  # 从租户配置而非模型猜测补齐货币。
    return adapted, {"from": "v1", "to": "v2", "currency_source": "tenant-profile"}  # 返回适配负载和 provenance。

strict_rows = []  # 收集按声明版本协商后的验证结果。
for request in requests:  # 遍历六个客户端请求。
    payload = request["payload"]  # 默认使用原始负载。
    adaptation = None  # 默认没有版本迁移。
    if request["client"] == "v1":  # 只有明确声明 v1 的客户端可使用兼容适配器。
        payload, adaptation = adapt_v1_to_v2(payload, tenant_currency="CNY")  # 从可信租户配置补齐货币。
    passed, errors = validate(payload, schemas["v2"])  # 用目标 v2 Schema 做完整验证。
    strict_rows.append({"id": request["id"], "passed": passed, "errors": errors, "payload": payload, "adaptation": adaptation})  # 保存兼容矩阵结果。
print("C1 适配结果：", strict_rows[0])  # 展示旧客户端如何显式迁移。

C1 适配结果： {'id': 'C1', 'passed': True, 'errors': [], 'payload': {'customer_id': 'U10', 'amount': 299.0, 'currency': 'CNY'}, 'adaptation': {'from': 'v1', 'to': 'v2', 'currency_source': 'tenant-profile'}}


## 结果解读：兼容矩阵与错误定位

In [4]:
expected_valid = {"C1": True, "C2": True, "C3": False, "C4": False, "C5": False, "C6": False}  # 定义六个契约样本的期望结论。
print("请求  client  朴素v2  严格协商  errors")  # 输出兼容矩阵表头。
for request, baseline, strict in zip(requests, baseline_rows, strict_rows):  # 对齐两种验证方案。
    print(f"{request['id']}   {request['client']:<6} {str(baseline['passed']):<8} {str(strict['passed']):<8} {strict['errors']}")  # 展示缺失、类型、未知和枚举错误。
baseline_correct = sum(row["passed"] == expected_valid[row["id"]] for row in baseline_rows) / len(requests)  # 计算朴素校验正确率。
strict_correct = sum(row["passed"] == expected_valid[row["id"]] for row in strict_rows) / len(requests)  # 计算严格协商正确率。
print(f"契约判定正确率 {baseline_correct:.1%} -> {strict_correct:.1%}")  # 量化版本协商和完整校验价值。
print("解读：C1 通过明确 v1 适配兼容；C3/C5/C6 在朴素必填检查下会穿透，严格校验给出具体失败字段。")  # 解释兼容而非宽松。

请求  client  朴素v2  严格协商  errors
C1   v1     False    True     []
C2   v2     True     True     []
C3   v2     True     False    ['type:amount']
C4   v2     False    False    ['missing:currency']
C5   v2     True     False    ['unknown:debug']
C6   v2     True     False    ['enum:currency']
契约判定正确率 33.3% -> 100.0%
解读：C1 通过明确 v1 适配兼容；C3/C5/C6 在朴素必填检查下会穿透，严格校验给出具体失败字段。


## 失败案例：对缺失货币静默默认 CNY

In [5]:
missing_currency_request = requests[3]  # 读取声明 v2 却缺失货币的 C4 请求。
unsafe_payload = missing_currency_request["payload"].copy()  # 复制请求以模拟服务端静默修改。
unsafe_payload["currency"] = "CNY"  # 未知租户背景下武断填入人民币。
unsafe_passed, unsafe_errors = validate(unsafe_payload, schemas["v2"])  # 静默默认后会通过结构校验。
safe_passed, safe_errors = validate(missing_currency_request["payload"], schemas["v2"])  # 正确做法是拒绝并要求调用方补充。
print(f"静默默认：payload={unsafe_payload} passed={unsafe_passed} errors={unsafe_errors}")  # 展示结构通过却可能业务记账错误。
print(f"严格拒绝：payload={missing_currency_request['payload']} passed={safe_passed} errors={safe_errors}")  # 展示可操作错误。
print("修正策略：只有 v1 契约和可信租户配置共同成立时才能适配；声明 v2 的缺失字段必须返回 schema error。")  # 总结默认值边界。

静默默认：payload={'customer_id': 'U13', 'amount': 120.0, 'currency': 'CNY'} passed=True errors=[]
严格拒绝：payload={'customer_id': 'U13', 'amount': 120.0} passed=False errors=['missing:currency']
修正策略：只有 v1 契约和可信租户配置共同成立时才能适配；声明 v2 的缺失字段必须返回 schema error。


### 生产边界与发布清单

In [6]:
release_manifest = {"tool": "invoice.create", "versions": ["v1", "v2"], "default": "v1", "v2_required": schemas["v2"]["required"], "compatibility_cases": len(requests), "rollout": "client-opt-in"}  # 构造工具版本发布清单。
print("发布清单：", release_manifest)  # 展示默认版本和灰度策略。
print("生产替换点：真实 MCP 还需标准 JSON Schema、initialize 能力协商、工具列表缓存失效、语义版本、遥测和旧版下线窗口。")  # 明确教学验证器边界。

发布清单： {'tool': 'invoice.create', 'versions': ['v1', 'v2'], 'default': 'v1', 'v2_required': ['customer_id', 'amount', 'currency'], 'compatibility_cases': 6, 'rollout': 'client-opt-in'}
生产替换点：真实 MCP 还需标准 JSON Schema、initialize 能力协商、工具列表缓存失效、语义版本、遥测和旧版下线窗口。


## 回归测试：最后只保护兼容适配与严格拒绝

In [7]:
assert strict_rows[0]["passed"] and strict_rows[0]["payload"]["currency"] == "CNY"  # 验证 v1 请求从可信租户配置完成适配。
assert strict_rows[1]["passed"] and strict_rows[1]["adaptation"] is None  # 验证合法 v2 请求无需改写。
assert strict_correct == 1.0 and strict_correct > baseline_correct  # 验证完整契约矩阵全部符合预期。
assert strict_rows[2]["errors"] == ["type:amount"] and strict_rows[4]["errors"] == ["unknown:debug"]  # 验证类型和未知字段错误可定位。
assert unsafe_passed and not safe_passed and safe_errors == ["missing:currency"]  # 验证静默默认反例与严格拒绝均发生。
print("回归测试通过：v1 适配、v2 直通、类型/未知字段、兼容矩阵和货币默认反例均成立。")  # 用少量断言总结 Schema 合同。

回归测试通过：v1 适配、v2 直通、类型/未知字段、兼容矩阵和货币默认反例均成立。
